In [2]:
from pandas.core.frame import DataFrame
import requests
import pandas as pd

In [3]:
urls = ["https://www.psl.noaa.gov/data/correlation/oni.data",
        "https://www.psl.noaa.gov/data/correlation/nina1.anom.data",
        "https://www.psl.noaa.gov/data/correlation/nina3.anom.data",
        "https://www.psl.noaa.gov/data/correlation/nina34.anom.data",
        "https://www.psl.noaa.gov/data/correlation/nina4.anom.data"]
labels = ["oni", "nino1", "nino3", "nino34", "nino4"]

datasets = []                                 # Conjunto de datasets recopilados de cada parámetro
años = []                                     # Primer y último año de cada parámetro

for url in urls:                              # Revisar cada URL
  text = requests.get(url).text.splitlines()  # Obtener líneas de texto
  years = text[0].split()                     # Obtener rango de años
  year, month, data = [], [], []              # Inicializar las variables a recopilar

  diff = int(years[1]) - int(years[0])
  for y in range(diff + 1):                   # Por cada año
    for m in range(1, 13):                    # y por cada mes
      year.append(int(years[0]) + y)
      month.append(m)
      data.append(text[y+1].split()[m])       # Recopilar un dato puntual

  datasets.append([year, month, data])        # Al terminar añadirlo a datasets
  años.append(int(years[0]))                  # Primer y último año de ese parámetro seleccionado
  años.append(int(years[1]))

year, month = [], []                          # Inicializar el tiempo
data = [[], [], [], [], []]                   # Crear casillas vacías

for y in range(min(años), max(años) + 1):     # Utilizar el rango años más extenso
  for m in range(1, 13):                      # Barrer cada mes
    month.append(m)                           # Rellenar los índices de año y mes
    year.append(y)
    data[0].append(0)                         # Rellenar las casillas vacías con ceros
    data[1].append(0)
    data[2].append(0)
    data[3].append(0)
    data[4].append(0)

for i in range(len(datasets)):                # Bucket sort de cada dataset recopilado
  for j in range(len(datasets[i][0])):
    data[i][datasets[i][1][j] + (datasets[i][0][j]-min(años))*12 - 1] = datasets[i][2][j]

df = pd.DataFrame({                           # Luego, crear y guardar el dataset procesado
    "year" : year, "month" : month,
    "oni" : data[0], "nino1" : data[1],
    "nino3" : data[2], "nino34" : data[3],
    "nino4" : data[4]
})

df.to_csv("enso_indices.csv", index=False)